# HybridRAG: Empirical Information Retrieval Benchmark

This notebook evaluates **Sparse (Okapi BM25)**, **Dense (FAISS `IndexFlatIP` + `all-MiniLM-L6-v2`)**, and **Hybrid (Reciprocal Rank Fusion)** retrieval across the 40-query PyTorch Golden Dataset (`data/evaluation/golden_queries.json`).

### Key Metrics Tracked
- **Recall@1, Recall@3, Recall@5**
- **Mean Reciprocal Rank (MRR)**
- **Precision@5**
- **End-to-End Query Latency (ms)** (Target `< 50ms`)

In [ ]:
import sys
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.evaluator import run_evaluation

report = run_evaluation()
summary = report['summary']
print(json.dumps(summary, indent=2))
print('Latency Summary:', report['latency'])

In [ ]:
modalities = ['Sparse (BM25)', 'Dense (FAISS)', 'Hybrid (RRF)']
keys = ['sparse', 'dense', 'hybrid']

recall_1 = [summary[k]['recall@1'] * 100 for k in keys]
recall_5 = [summary[k]['recall@5'] * 100 for k in keys]
mrr_scores = [summary[k]['mrr'] * 100 for k in keys]

x = np.arange(len(modalities))
width = 0.25

fig, ax = plt.subplots(figsize=(10, 5.5))
b1 = ax.bar(x - width, recall_1, width, label='Recall@1 (%)', color='#3b82f6')
b2 = ax.bar(x, recall_5, width, label='Recall@5 (%)', color='#10b981')
b3 = ax.bar(x + width, mrr_scores, width, label='MRR (x100)', color='#8b5cf6')

ax.set_ylabel('Score (%)')
ax.set_title('HybridRAG Retrieval Benchmark: Sparse (BM25) vs Dense (FAISS) vs Hybrid (RRF)')
ax.set_xticks(x)
ax.set_xticklabels(modalities)
ax.set_ylim(0, 110)
ax.legend()
ax.grid(axis='y', linestyle='--', alpha=0.5)

for bars in (b1, b2, b3):
    for bar in bars:
        h = bar.get_height()
        ax.annotate(f'{h:.1f}%', xy=(bar.get_x() + bar.get_width() / 2, h),
                    xytext=(0, 4), textcoords='offset points', ha='center', fontsize=9)

plt.tight_layout()
plt.show()